# Replay Agent Timing EDA

Analyze model startup time and mean subsequent action time for teams in the newest replay archive.

This notebook:

- reads the newest `M.D.zip` archive without extracting it;
- caches one row per replay player under `data/replay_timing/`;
- aggregates timing at team level;
- compares all teams with a configurable score-filtered cohort;
- fits one global four-cluster K-Means model and reuses it for the filtered cohort.

## Context & Methods

The replay engine starts each player with 600 seconds of overage time. A positive decrease in `remainingOverageTime` represents time consumed by an agent call.

- **Startup time:** the first positive decrease for a player, normally the initial deck submission and model loading call.
- **Mean subsequent action time:** all later positive decreases, pooled across a team's replays and divided by the total number of later calls.
- **Team grain:** every plotted row is one exact `TeamNames` value.
- **Score cohort:** a replay-level filter because the manifest does not map its two individual scores back to player indices.
- **Clusters:** behavioral timing profiles only; they do not prove implementation type.

In [ ]:
from __future__ import annotations

import io
import json
import math
import re
import zipfile
from pathlib import Path, PurePosixPath

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

FORCE_REBUILD = False
SCORE_THRESHOLD = 1100.0
SCORE_MODE = "avg"  # avg, min, or max
N_CLUSTERS = 4
RANDOM_STATE = 42
MAX_ERROR_EXAMPLES = 20
PROGRESS_EVERY = 500

if SCORE_MODE not in {"avg", "min", "max"}:
    raise ValueError("SCORE_MODE must be 'avg', 'min', or 'max'")


def find_imitation_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if base.name == "imitation_learning" and (base / "training").is_dir():
            return base
        candidate = base / "imitation_learning"
        if (candidate / "training").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate imitation_learning from the current working directory"
    )


PROJECT_ROOT = find_imitation_root()
REPOSITORY_ROOT = PROJECT_ROOT.parent
REPLAY_ROOT = REPOSITORY_ROOT / "replay_episodes"
OUTPUT_DIR = PROJECT_ROOT / "data" / "replay_timing"

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

print(f"project_root={PROJECT_ROOT}")
print(f"replay_root={REPLAY_ROOT}")

### Locate Latest Replay Archive

Dates are parsed numerically, so `7.10.zip` correctly sorts after `7.9.zip`.

In [ ]:
DATE_RE = re.compile(r"^(\d{1,2})\.(\d{1,2})\.zip$")


def find_latest_archive(replay_root: Path) -> tuple[str, Path]:
    candidates: list[tuple[tuple[int, int], str, Path]] = []
    for path in replay_root.glob("*.zip"):
        match = DATE_RE.fullmatch(path.name)
        if match is None:
            continue
        month, day = map(int, match.groups())
        if not (1 <= month <= 12 and 1 <= day <= 31):
            continue
        candidates.append(((month, day), f"{month}.{day}", path))

    if not candidates:
        raise FileNotFoundError(
            f"No replay archives named M.D.zip were found in {replay_root}"
        )

    _, date_label, archive_path = max(candidates, key=lambda item: item[0])
    return date_label, archive_path


DATE_LABEL, ARCHIVE_PATH = find_latest_archive(REPLAY_ROOT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLAYER_CACHE_PATH = OUTPUT_DIR / f"{DATE_LABEL}.player_timings.csv"
TEAM_OUTPUT_PATH = OUTPUT_DIR / f"{DATE_LABEL}.team_timings.csv"

print(f"selected_archive={ARCHIVE_PATH.name}")
print(f"player_cache={PLAYER_CACHE_PATH}")
print(f"team_output={TEAM_OUTPUT_PATH}")

### Load Manifest

The active score column is selected from `avg_score`, `min_score`, or the reconstructed `max_score = sum_score - min_score`.

In [ ]:
MANIFEST_COLUMNS = {
    "episode_id",
    "avg_score",
    "min_score",
    "sum_score",
    "agent_count",
}


def find_manifest_member(archive: zipfile.ZipFile) -> str:
    names = [
        name
        for name in archive.namelist()
        if PurePosixPath(name).name == "manifest.csv"
    ]
    if len(names) != 1:
        raise ValueError(
            f"{archive.filename} must contain exactly one manifest.csv; "
            f"found {len(names)}"
        )
    return names[0]


def load_manifest(archive_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(archive_path) as archive:
        member = find_manifest_member(archive)
        with archive.open(member) as raw:
            manifest = pd.read_csv(
                io.TextIOWrapper(raw, encoding="utf-8-sig", newline=""),
                dtype={"episode_id": "string"},
            )

    missing = MANIFEST_COLUMNS - set(manifest.columns)
    if missing:
        raise ValueError(f"manifest.csv is missing columns: {sorted(missing)}")

    manifest = manifest.copy()
    manifest["episode_id"] = manifest["episode_id"].str.strip()
    if manifest["episode_id"].isna().any() or manifest["episode_id"].eq("").any():
        raise ValueError("manifest.csv contains empty episode_id values")
    if manifest["episode_id"].duplicated().any():
        duplicate = manifest.loc[
            manifest["episode_id"].duplicated(), "episode_id"
        ].iloc[0]
        raise ValueError(f"manifest.csv contains duplicate episode_id={duplicate}")

    numeric_columns = ["avg_score", "min_score", "sum_score", "agent_count"]
    for column in numeric_columns:
        manifest[column] = pd.to_numeric(manifest[column], errors="raise")

    if not np.isfinite(manifest[["avg_score", "min_score", "sum_score"]]).all().all():
        raise ValueError("manifest.csv contains non-finite scores")
    if not manifest["agent_count"].eq(2).all():
        raise ValueError("This notebook requires exactly two agents per replay")

    manifest["max_score"] = manifest["sum_score"] - manifest["min_score"]
    if (manifest["max_score"] < manifest["min_score"]).any():
        raise ValueError("manifest.csv has inconsistent min_score and sum_score")

    manifest["score_value"] = manifest[f"{SCORE_MODE}_score"]
    manifest["passes_score_filter"] = (
        manifest["score_value"] > SCORE_THRESHOLD
    )
    return manifest


manifest = load_manifest(ARCHIVE_PATH)
manifest.head()

### Extract Replay-Player Timings

An empty action can still be a valid engine call, so timing is detected from positive overage-time decreases rather than action contents.

In [ ]:
def finite_remaining_time(agent_state: object) -> float | None:
    if not isinstance(agent_state, dict):
        return None
    observation = agent_state.get("observation")
    if not isinstance(observation, dict):
        return None
    value = observation.get("remainingOverageTime")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return None
    value = float(value)
    return value if math.isfinite(value) else None


def extract_player_timing(
    payload: dict[str, object],
    episode_id: str,
) -> list[dict[str, object]]:
    info = payload.get("info")
    if not isinstance(info, dict):
        return []
    team_names = info.get("TeamNames")
    steps = payload.get("steps")
    if not isinstance(team_names, list) or not isinstance(steps, list):
        return []

    rows: list[dict[str, object]] = []
    for player_index in range(min(2, len(team_names))):
        team_name = str(team_names[player_index] or "").strip()
        if not team_name:
            continue

        remaining_values: list[float] = []
        for step in steps:
            if not isinstance(step, list) or player_index >= len(step):
                continue
            value = finite_remaining_time(step[player_index])
            if value is not None:
                remaining_values.append(value)

        if len(remaining_values) < 2:
            continue

        positive_deltas = [
            previous - current
            for previous, current in zip(
                remaining_values, remaining_values[1:]
            )
            if previous - current > 1e-9
        ]
        if not positive_deltas:
            continue

        startup_time = positive_deltas[0]
        later_times = positive_deltas[1:]
        later_total = float(sum(later_times))
        later_count = len(later_times)
        rows.append(
            {
                "episode_id": str(episode_id),
                "player_index": player_index,
                "team_name": team_name,
                "startup_time_seconds": float(startup_time),
                "subsequent_time_seconds": later_total,
                "subsequent_action_count": later_count,
                "mean_step_time_seconds": (
                    later_total / later_count if later_count else np.nan
                ),
            }
        )
    return rows


synthetic_payload = {
    "info": {"TeamNames": ["A", "B"]},
    "steps": [
        [
            {"observation": {"remainingOverageTime": 600.0}},
            {"observation": {"remainingOverageTime": 600.0}},
        ],
        [
            {"observation": {"remainingOverageTime": 598.0}},
            {"observation": {"remainingOverageTime": 599.0}},
        ],
        [
            {"observation": {"remainingOverageTime": 597.5}},
            {"observation": {"remainingOverageTime": 598.75}},
        ],
    ],
}
synthetic = extract_player_timing(synthetic_payload, "synthetic")
assert synthetic[0]["startup_time_seconds"] == 2.0
assert synthetic[0]["subsequent_time_seconds"] == 0.5
assert synthetic[0]["subsequent_action_count"] == 1
assert synthetic[1]["startup_time_seconds"] == 1.0
assert synthetic[1]["mean_step_time_seconds"] == 0.25
print("Synthetic timing checks passed.")

In [ ]:
PLAYER_COLUMNS = [
    "date",
    "episode_id",
    "player_index",
    "team_name",
    "startup_time_seconds",
    "subsequent_time_seconds",
    "subsequent_action_count",
    "mean_step_time_seconds",
    "avg_score",
    "min_score",
    "max_score",
    "sum_score",
    "score_value",
    "passes_score_filter",
]


def apply_score_filter(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    score_column = f"{SCORE_MODE}_score"
    if score_column not in frame:
        raise ValueError(f"Player cache is missing {score_column}")
    frame["score_value"] = pd.to_numeric(
        frame[score_column], errors="raise"
    )
    frame["passes_score_filter"] = (
        frame["score_value"] > SCORE_THRESHOLD
    )
    return frame


def build_player_timings(
    archive_path: Path,
    manifest_frame: pd.DataFrame,
) -> tuple[pd.DataFrame, list[dict[str, str]], int]:
    manifest_lookup = manifest_frame.set_index("episode_id").to_dict("index")
    rows: list[dict[str, object]] = []
    errors: list[dict[str, str]] = []
    failed_count = 0

    with zipfile.ZipFile(archive_path) as archive:
        members = [
            member
            for member in archive.infolist()
            if not member.is_dir()
            and PurePosixPath(member.filename).suffix.lower() == ".json"
        ]
        total = len(members)
        for member_index, member in enumerate(members, start=1):
            try:
                with archive.open(member) as raw:
                    payload = json.load(
                        io.TextIOWrapper(raw, encoding="utf-8")
                    )
                payload_info = payload.get("info") or {}
                episode_id = str(
                    payload_info.get("EpisodeId")
                    or PurePosixPath(member.filename).stem
                )
                manifest_values = manifest_lookup.get(episode_id)
                if manifest_values is None:
                    raise KeyError(
                        f"episode_id={episode_id} is absent from manifest.csv"
                    )
                extracted = extract_player_timing(payload, episode_id)
                if not extracted:
                    raise ValueError("no valid player timing rows")
                for row in extracted:
                    rows.append(
                        {
                            "date": DATE_LABEL,
                            **row,
                            "avg_score": float(
                                manifest_values["avg_score"]
                            ),
                            "min_score": float(
                                manifest_values["min_score"]
                            ),
                            "max_score": float(
                                manifest_values["max_score"]
                            ),
                            "sum_score": float(
                                manifest_values["sum_score"]
                            ),
                        }
                    )
            except Exception as exc:
                failed_count += 1
                if len(errors) < MAX_ERROR_EXAMPLES:
                    errors.append(
                        {
                            "member": member.filename,
                            "error": f"{type(exc).__name__}: {exc}",
                        }
                    )
            if PROGRESS_EVERY and member_index % PROGRESS_EVERY == 0:
                print(
                    f"processed={member_index:,}/{total:,} "
                    f"rows={len(rows):,} failed={failed_count:,}",
                    flush=True,
                )

    result = pd.DataFrame(rows)
    if result.empty:
        raise ValueError("No valid replay-player timing rows were extracted")
    result = apply_score_filter(result)
    return result[PLAYER_COLUMNS], errors, failed_count

### Load or Build Player Timing Cache

Changing the score mode or threshold does not require rescanning the ZIP: raw manifest scores are retained and the active filter is recomputed after loading.

In [ ]:
cache_was_used = False
extraction_errors: list[dict[str, str]] = []
failed_member_count = 0

if PLAYER_CACHE_PATH.exists() and not FORCE_REBUILD:
    cached = pd.read_csv(
        PLAYER_CACHE_PATH,
        dtype={"episode_id": "string", "team_name": "string"},
    )
    missing_cache_columns = set(PLAYER_COLUMNS) - set(cached.columns)
    raw_score_columns = {"avg_score", "min_score", "max_score", "sum_score"}
    if missing_cache_columns - {"score_value", "passes_score_filter"}:
        print(
            "Cache schema is incomplete; rebuilding from the replay archive."
        )
        player_timings, extraction_errors, failed_member_count = (
            build_player_timings(ARCHIVE_PATH, manifest)
        )
    elif not raw_score_columns.issubset(cached.columns):
        print(
            "Cache lacks raw score columns; rebuilding from the replay archive."
        )
        player_timings, extraction_errors, failed_member_count = (
            build_player_timings(ARCHIVE_PATH, manifest)
        )
    else:
        player_timings = apply_score_filter(cached)
        player_timings = player_timings[PLAYER_COLUMNS]
        cache_was_used = True
else:
    player_timings, extraction_errors, failed_member_count = (
        build_player_timings(ARCHIVE_PATH, manifest)
    )

player_timings.to_csv(PLAYER_CACHE_PATH, index=False)
print(
    f"cache_used={cache_was_used} rows={len(player_timings):,} "
    f"failed_members={failed_member_count:,}"
)
if extraction_errors:
    display(pd.DataFrame(extraction_errors))

### Validate Extracted Timings

In [ ]:
assert not player_timings.empty
assert player_timings["team_name"].astype(str).str.strip().ne("").all()
assert player_timings["startup_time_seconds"].ge(0).all()
assert player_timings["subsequent_time_seconds"].ge(0).all()
assert player_timings["subsequent_action_count"].ge(0).all()
assert player_timings["player_index"].isin([0, 1]).all()
assert player_timings["episode_id"].notna().all()

extraction_summary = pd.DataFrame(
    {
        "metric": [
            "date",
            "player rows",
            "episodes",
            "teams",
            "score filter",
            "filtered player rows",
            "failed replay members",
        ],
        "value": [
            DATE_LABEL,
            f"{len(player_timings):,}",
            f"{player_timings['episode_id'].nunique():,}",
            f"{player_timings['team_name'].nunique():,}",
            f"{SCORE_MODE}_score > {SCORE_THRESHOLD:g}",
            f"{player_timings['passes_score_filter'].sum():,}",
            f"{failed_member_count:,}",
        ],
    }
)
display(extraction_summary)

## Results

### Team Aggregation

Startup time is averaged across replay appearances. Subsequent action time is pooled across all later calls, so the reported team mean is action-weighted rather than an unweighted mean of replay means.

In [ ]:
def aggregate_teams(
    player_frame: pd.DataFrame,
    mask: pd.Series,
    cohort: str,
) -> pd.DataFrame:
    selected = player_frame.loc[mask].copy()
    grouped = selected.groupby("team_name", as_index=False).agg(
        replay_count=("episode_id", "nunique"),
        startup_time_mean_seconds=("startup_time_seconds", "mean"),
        subsequent_time_seconds=("subsequent_time_seconds", "sum"),
        subsequent_action_count=("subsequent_action_count", "sum"),
    )
    grouped = grouped[
        grouped["subsequent_action_count"] > 0
    ].copy()
    grouped["mean_step_time_seconds"] = (
        grouped["subsequent_time_seconds"]
        / grouped["subsequent_action_count"]
    )
    grouped.insert(0, "cohort", cohort)
    return grouped


all_mask = pd.Series(True, index=player_timings.index)
filtered_mask = player_timings["passes_score_filter"].astype(bool)

all_teams = aggregate_teams(player_timings, all_mask, "all")
filtered_teams = aggregate_teams(
    player_timings, filtered_mask, "score_filtered"
)

expected_time = player_timings["subsequent_time_seconds"].sum()
observed_time = all_teams["subsequent_time_seconds"].sum()
assert np.isclose(expected_time, observed_time)
assert all_teams["team_name"].is_unique
assert filtered_teams["team_name"].is_unique

print(f"all_teams={len(all_teams):,}")
print(f"filtered_teams={len(filtered_teams):,}")

### Global Timing Clusters

K-Means is fitted once on all teams after `log1p` and standardization. The score-filtered cohort receives predictions from the same scaler and cluster centers.

In [ ]:
FEATURE_COLUMNS = [
    "startup_time_mean_seconds",
    "mean_step_time_seconds",
]


def log_features(frame: pd.DataFrame) -> np.ndarray:
    values = frame[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Timing features must be finite and nonnegative")
    return np.log1p(values)


if len(all_teams) < N_CLUSTERS:
    raise ValueError(
        f"Need at least {N_CLUSTERS} all-cohort teams; "
        f"found {len(all_teams)}"
    )

scaler = StandardScaler()
all_scaled = scaler.fit_transform(log_features(all_teams))
kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10,
)
all_teams["cluster"] = kmeans.fit_predict(all_scaled)

if len(filtered_teams):
    filtered_scaled = scaler.transform(log_features(filtered_teams))
    filtered_teams["cluster"] = kmeans.predict(filtered_scaled)
else:
    filtered_teams["cluster"] = np.array([], dtype=np.int64)

cluster_centers_seconds = np.expm1(
    scaler.inverse_transform(kmeans.cluster_centers_)
)

if len(filtered_teams):
    repeated_labels = kmeans.predict(
        scaler.transform(log_features(filtered_teams))
    )
    assert np.array_equal(
        repeated_labels,
        filtered_teams["cluster"].to_numpy(),
    )
assert len(cluster_centers_seconds) == N_CLUSTERS

team_timings = pd.concat(
    [all_teams, filtered_teams],
    ignore_index=True,
)
team_timings["score_mode"] = SCORE_MODE
team_timings["score_threshold"] = SCORE_THRESHOLD
team_timings = team_timings.sort_values(
    ["cohort", "cluster", "team_name"],
    kind="stable",
).reset_index(drop=True)
team_timings.to_csv(TEAM_OUTPUT_PATH, index=False)

print(f"wrote {TEAM_OUTPUT_PATH} ({len(team_timings):,} rows)")

In [ ]:
cohort_summary = (
    team_timings.groupby("cohort", as_index=False)
    .agg(
        team_count=("team_name", "nunique"),
        total_replays=("replay_count", "sum"),
        total_actions=("subsequent_action_count", "sum"),
        median_startup_seconds=(
            "startup_time_mean_seconds",
            "median",
        ),
        median_step_seconds=("mean_step_time_seconds", "median"),
    )
)

center_frame = pd.DataFrame(
    {
        "cluster": np.arange(N_CLUSTERS, dtype=int),
        "center_startup_seconds": cluster_centers_seconds[:, 0],
        "center_step_seconds": cluster_centers_seconds[:, 1],
    }
)
all_cluster_counts = all_teams["cluster"].value_counts()
filtered_cluster_counts = filtered_teams["cluster"].value_counts()
center_frame["all_team_count"] = (
    center_frame["cluster"].map(all_cluster_counts).fillna(0).astype(int)
)
center_frame["filtered_team_count"] = (
    center_frame["cluster"]
    .map(filtered_cluster_counts)
    .fillna(0)
    .astype(int)
)

display(cohort_summary)
display(center_frame.round(6))

### Timing Distributions

Histogram y-axes count unique teams. Timing axes use seconds and logarithmic scaling to preserve long-tail structure.

In [ ]:
CLUSTER_COLORS = {
    0: "#2F6BFF",
    1: "#D97706",
    2: "#708238",
    3: "#C24170",
}
BASE_COLOR = "#2F6BFF"
FILTER_COLOR = "#D97706"
FILTER_LABEL = f"{SCORE_MODE}_score > {SCORE_THRESHOLD:g}"


def plot_histogram(
    frame: pd.DataFrame,
    column: str,
    title: str,
    xlabel: str,
    color: str,
) -> None:
    values = frame.loc[frame[column] > 0, column]
    if values.empty:
        print(f"No positive values available for: {title}")
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(
        values,
        bins=35,
        color=color,
        edgecolor="white",
        linewidth=0.6,
        ax=ax,
    )
    ax.set_xscale("log")
    ax.set_title(title, loc="left")
    ax.text(
        0,
        1.01,
        f"Unique teams: {len(values):,} | Unit: seconds | Log-scaled x-axis",
        transform=ax.transAxes,
        color="#5B6472",
        fontsize=10,
        va="bottom",
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Teams")
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_histogram(
    all_teams,
    "startup_time_mean_seconds",
    "All Teams: Startup Time Distribution",
    "Mean startup time (seconds)",
    BASE_COLOR,
)

plot_histogram(
    all_teams,
    "mean_step_time_seconds",
    "All Teams: Mean Subsequent Action Time Distribution",
    "Mean subsequent action time (seconds)",
    BASE_COLOR,
)

In [ ]:
if filtered_teams.empty:
    print(f"No teams satisfy {FILTER_LABEL}.")
else:
    plot_histogram(
        filtered_teams,
        "startup_time_mean_seconds",
        f"Score-Filtered Teams: Startup Time Distribution ({FILTER_LABEL})",
        "Mean startup time (seconds)",
        FILTER_COLOR,
    )

    plot_histogram(
        filtered_teams,
        "mean_step_time_seconds",
        (
            "Score-Filtered Teams: Mean Subsequent Action Time "
            f"Distribution ({FILTER_LABEL})"
        ),
        "Mean subsequent action time (seconds)",
        FILTER_COLOR,
    )

### Startup vs Mean Subsequent Action Time

Each point is one team. Black X markers are the centers fitted from the all-team cohort.

In [ ]:
def plot_cluster_scatter(
    frame: pd.DataFrame,
    title: str,
    subtitle: str,
) -> None:
    if frame.empty:
        print(f"No rows available for: {title}")
        return

    plotted = frame[
        (frame["mean_step_time_seconds"] > 0)
        & (frame["startup_time_mean_seconds"] > 0)
    ].copy()
    if plotted.empty:
        print(f"No positive timing pairs available for: {title}")
        return

    fig, ax = plt.subplots(figsize=(10, 7))
    for cluster in range(N_CLUSTERS):
        cluster_rows = plotted[plotted["cluster"] == cluster]
        if cluster_rows.empty:
            continue
        ax.scatter(
            cluster_rows["mean_step_time_seconds"],
            cluster_rows["startup_time_mean_seconds"],
            s=38,
            alpha=0.72,
            color=CLUSTER_COLORS[cluster],
            label=f"Cluster {cluster}",
            edgecolors="white",
            linewidths=0.35,
        )

    ax.scatter(
        cluster_centers_seconds[:, 1],
        cluster_centers_seconds[:, 0],
        s=190,
        marker="X",
        color="#111827",
        edgecolors="white",
        linewidths=0.9,
        label="Global centers",
        zorder=5,
    )
    for cluster, (startup_center, step_center) in enumerate(
        cluster_centers_seconds
    ):
        ax.annotate(
            str(cluster),
            (step_center, startup_center),
            xytext=(7, 6),
            textcoords="offset points",
            color="#111827",
            fontsize=10,
            weight="bold",
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(title, loc="left")
    ax.text(
        0,
        1.01,
        subtitle,
        transform=ax.transAxes,
        color="#5B6472",
        fontsize=10,
        va="bottom",
    )
    ax.set_xlabel("Mean subsequent action time (seconds)")
    ax.set_ylabel("Mean startup time (seconds)")
    ax.legend(frameon=False, ncol=3, loc="best")
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()


plot_cluster_scatter(
    all_teams,
    "All Teams: Startup vs Mean Subsequent Action Time",
    (
        f"Teams: {len(all_teams):,} | Global K-Means ({N_CLUSTERS} clusters) "
        "| Log-scaled axes"
    ),
)

In [ ]:
if filtered_teams.empty:
    print(f"No teams satisfy {FILTER_LABEL}.")
else:
    plot_cluster_scatter(
        filtered_teams,
        (
            "Score-Filtered Teams: Startup vs Mean Subsequent "
            f"Action Time ({FILTER_LABEL})"
        ),
        (
            f"Teams: {len(filtered_teams):,} | Assigned with global centers "
            "| Log-scaled axes"
        ),
    )

## Takeaways and Caveats

After execution, use the six figures and the two summary tables above to describe the observed timing profiles.

Important limitations:

- Cluster membership describes timing behavior; it does not prove that a team uses rules, a neural network, search, or a hybrid.
- `avg` and `max` score modes select whole replays and may include a participant whose individual score is below the threshold.
- `min` mode guarantees both replay participants exceed the threshold.
- Teams with few replays or few subsequent actions have noisier estimates; inspect `replay_count` and `subsequent_action_count` in the exported team CSV.
- Startup time includes the first measured agent call and is an operational proxy for initialization/model loading time.